1. 共用參數設定


In [ ]:
'''
Measure the elapsed time of every code cell, including cells that raise errors.
記錄每個 Code Cell 的執行時間，即使發生錯誤也顯示已耗費的時間。
'''
from time import perf_counter as _cell_perf_counter
from IPython import get_ipython as _cell_get_ipython

_CELL_TIMER_SHELL = _cell_get_ipython()
if _CELL_TIMER_SHELL is None:
    raise RuntimeError('請在 Jupyter Notebook 中執行本教材。')
_CELL_TIMER_STATE = {'started': _cell_perf_counter()}

def _cell_timer_begin(_info):
    _CELL_TIMER_STATE['started'] = _cell_perf_counter()

def _cell_timer_end(result):
    started = _CELL_TIMER_STATE.pop('started', None)
    if started is None:
        return
    minutes, seconds = divmod(_cell_perf_counter() - started, 60)
    elapsed = f'{int(minutes)} 分 {seconds:05.2f} 秒' if minutes else f'{seconds:.2f} 秒'
    status = '執行失敗' if result.error_before_exec or result.error_in_exec else '執行完成'
    print(f'⏱ 本格{status}，耗時：{elapsed}')

'''
Replace existing timers when the first cell is executed again.
重新執行第一格時，先取消舊計時器，避免重複顯示耗時。
'''
for _event_name, _callback in globals().get('_CELL_TIMER_CALLBACKS', ()):
    try:
        _CELL_TIMER_SHELL.events.unregister(_event_name, _callback)
    except ValueError:
        pass
_CELL_TIMER_CALLBACKS = (
    ('pre_run_cell', _cell_timer_begin),
    ('post_run_cell', _cell_timer_end),
)
for _event_name, _callback in _CELL_TIMER_CALLBACKS:
    _CELL_TIMER_SHELL.events.register(_event_name, _callback)

'''
Import standard-library tools for editable paths and local service settings.
匯入設定路徑與本機服務參數所需的 Python 內建工具。
'''
import os
from pathlib import Path
ROOT = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path.home() / 'ai_course'
ROOT = ROOT.expanduser().resolve()  # Ubuntu 可改成 Path('/home/your_user/ai_course')
SCRATCH_ROOT = Path('/tmp') if Path('/kaggle/working').is_dir() else ROOT / 'cache'
SCRATCH_ROOT = SCRATCH_ROOT.expanduser().resolve()  # 可改成其他儲存裝置或模型磁碟
ROOT.mkdir(parents=True, exist_ok=True)
SCRATCH_ROOT.mkdir(parents=True, exist_ok=True)

REPO = ROOT / 'heartlib'
GPU_COUNT = 2  # Kaggle T4 ×2；換主機時依 NVIDIA GPU 數量設定（1～4）
MODEL_GPU_ID = 0  # HeartMuLa
AUX_GPU_ID = 1  # HeartCodec

'''
Select the downloaded HeartMuLa and codec checkpoint directory.
設定 HeartMuLa 主模型與 Codec 權重的下載位置。
'''
CKPT = SCRATCH_ROOT / 'heartmula/ckpt'
HF_CACHE_DIR = SCRATCH_ROOT / 'heartmula/huggingface'
os.environ['HF_HOME'] = str(HF_CACHE_DIR)
os.environ['HF_HUB_CACHE'] = str(HF_CACHE_DIR / 'hub')
os.environ['HF_XET_CHUNK_CACHE_SIZE_BYTES'] = '0'

'''
Select the optional remote FIFO API port; change only if the port conflicts.
指定選用 FIFO 遠端 API 的本機連接埠；只有發生 Port 衝突時才需要修改。
'''
REMOTE_PORT = 8765  # 例如 8765、8766，範圍 1024～65535
REMOTE_SERVICE_ID = '06_HeartMuLa_音樂生成_Kaggle'
REMOTE_ROOT = ROOT / 'remote_api_jobs' / REMOTE_SERVICE_ID

'''
Limit how many remote jobs wait for the single GPU worker.
設定 FIFO 最多等待的工作數量；超出時回傳 HTTP 429，client 會延遲重試。
'''
REMOTE_QUEUE_CAPACITY = 32  # 建議 12～32，依課堂人數調整
REMOTE_UPLOAD_CONCURRENCY = 2  # 建議 1～2
REMOTE_RESULT_TTL_HOURS = 24
REMOTE_RESULT_LIMIT_GIB = 8
REMOTE_MIN_FREE_GIB = 2

'''
Mark exceptionally long remote inference as stalled, without forcibly starting a second GPU job.
設定遠端推論超時警戒值；卡住時不強行啟動第二筆 GPU 推論。
'''
REMOTE_STALL_HOURS = float('inf')  # 不限制單次生成時長，也不將執行中的工作誤判為停滯
assert 1024 <= REMOTE_PORT <= 65535, 'REMOTE_PORT 必須介於 1024～65535。'
assert 1 <= REMOTE_QUEUE_CAPACITY <= 1024, 'FIFO 佇列容量設定無效。'
assert 1 <= REMOTE_UPLOAD_CONCURRENCY <= 8, '上傳並發限制設定無效。'
assert REMOTE_RESULT_TTL_HOURS > 0 and REMOTE_RESULT_LIMIT_GIB > 0 and REMOTE_MIN_FREE_GIB >= 0
assert REMOTE_STALL_HOURS > 0
print('✓ 第 1 節完成：共用參數已設定，請繼續執行環境安裝。')

assert 1 <= GPU_COUNT <= 4, 'GPU_COUNT 必須介於 1～4。'
assert all(0 <= gpu < GPU_COUNT for gpu in (MODEL_GPU_ID, AUX_GPU_ID)), 'GPU ID 超出 GPU_COUNT 的範圍。'
assert GPU_COUNT != 1 or MODEL_GPU_ID == AUX_GPU_ID, '單張 GPU 時，兩種元件應使用相同的 GPU ID。'


2. 安裝與 GPU 檢查


In [ ]:
'''
Import the libraries needed for this section.
匯入本段程式所需的函式庫。
'''
import os
import subprocess
import sys
from pathlib import Path
import torch
assert torch.cuda.is_available() and torch.cuda.device_count() >= GPU_COUNT, '可用 CUDA GPU 數量不足；請檢查 NVIDIA 驅動、PyTorch 與 GPU_COUNT。'
CKPT.mkdir(parents=True, exist_ok=True)

'''
Keep the Hugging Face cache on temporary storage.
將 Hugging Face 快取放到暫存空間。
'''
os.environ['HF_HOME'] = str(HF_CACHE_DIR)
os.environ['HF_XET_CACHE'] = str(HF_CACHE_DIR / 'xet')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

'''
Download the official source if it is not already available.
只有尚未存在官方原始碼時才下載。
'''
if not (REPO / 'pyproject.toml').exists():
    subprocess.run(['git', 'clone', '-q', 'https://github.com/HeartMuLa/heartlib.git', str(REPO)], check=True)
else:
    assert (REPO / '.git').exists(), 'HeartMuLa 路徑不是 Git 倉庫，請先檢查。'
    changes = subprocess.check_output(['git', '-C', str(REPO), 'status', '--porcelain'], text=True).strip()

    '''
    Update only when doing so will not overwrite customized files.
    只有不會覆寫使用者修改時，才同步上游程式。
    '''
    if not changes:
        subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', '-q'], check=True)
    else:
        print('HeartMuLa 原始碼已有本機修改，為避免衝突未自動更新；建議使用全新 Kaggle Session。')

print('HeartMuLa commit:', subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', '--short', 'HEAD'], text=True).strip())
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'transformers==4.57.0', 'tokenizers==0.22.1',
    'huggingface-hub==0.36.2', 'vector-quantize-pytorch==1.27.15', 'soundfile',
], check=True)

'''
Keep Kaggle's PyTorch and TorchAO when installing TorchTune.
安裝 TorchTune 時保留 Kaggle 的 PyTorch 與 TorchAO。
'''
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', 'torchtune==0.4.0'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '-e', str(REPO)], check=True)
sys.path.insert(0, str(REPO / 'src'))

'''
Give installation checks the same import priority in their subprocesses.
讓安裝檢查子行程也優先載入正式套件，而非外層 Git 目錄。
'''
os.environ['PYTHONPATH'] = str(REPO / 'src') + os.pathsep + os.environ.get('PYTHONPATH', '')
subprocess.run([
    sys.executable, '-c',
    'import torchao, torchtune, torchaudio; from heartlib import HeartMuLaGenPipeline',
], check=True)
for gpu in sorted({MODEL_GPU_ID, AUX_GPU_ID}):
    print(f'GPU {gpu}: {torch.cuda.get_device_name(gpu)}')

print('✓ 第 2 節完成：環境安裝與雙 GPU 檢查已完成。')


3. 官方模型與 RoPE 相容修正


In [ ]:
'''
Import the official Hugging Face model snapshot downloader.
匯入 Hugging Face 官方模型下載工具。
'''
from huggingface_hub import snapshot_download
for model_id, directory, patterns in [
    ('HeartMuLa/HeartMuLaGen', CKPT, ['tokenizer.json', 'gen_config.json']),
    ('HeartMuLa/HeartMuLa-oss-3B-happy-new-year', CKPT / 'HeartMuLa-oss-3B', ['config.json', '*.safetensors', '*.safetensors.index.json']),
    ('HeartMuLa/HeartCodec-oss-20260123', CKPT / 'HeartCodec-oss', ['config.json', '*.safetensors', '*.safetensors.index.json']),
]:
    snapshot_download(repo_id=model_id, local_dir=str(directory), allow_patterns=patterns, max_workers=2)

model_file = REPO / 'src/heartlib/heartmula/modeling_heartmula.py'

'''
Read the source before applying the known RoPE compatibility patch.
套用已知 RoPE 相容修正前先讀取原始碼。
'''
source = model_file.read_text(encoding='utf-8')
original = '        with device:\n            self.backbone.setup_caches(max_batch_size, dtype)'
replacement = (
    "        with device:\n"
    "            # Restore RoPE caches on the actual compute device.\n"
    "            # 在實際運算裝置上重建 RoPE 快取。\n"
    "            for module in self.modules():\n"
    "                # Repair position embeddings only when their cache is missing.\n"
    "                # 只修復尚未建立快取的位置 embedding。\n"
    "                if hasattr(module, 'rope_init') and not getattr(module, 'is_cache_built', True):\n"
    "                    # Initialize the missing positional cache.\n"
    "                    # 初始化缺少的位置快取。\n"
    "                    module.rope_init()\n"
    "                    # Keep the positional module on the model GPU.\n"
    "                    # 確保位置模組維持在模型所使用的 GPU。\n"
    "                    module.to(device)\n"
    "            self.backbone.setup_caches(max_batch_size, dtype)"
)

'''
Avoid applying the compatibility fix twice.
避免重複套用相容修正。
'''
if 'module.rope_init()' not in source:
    assert original in source, 'HeartMuLa 的 setup_caches 已變更，請檢查官方版本。'
    model_file.write_text(source.replace(original, replacement, 1), encoding='utf-8')

print('✓ 第 3 節完成：模型相容設定已完成。')


4. 載入模型（僅首次）


In [ ]:
'''
Import the libraries needed for this section.
匯入本段程式所需的函式庫。
'''
import sys
from pathlib import Path
import torch
sys.path.insert(0, str(REPO / 'src'))
cached_heartlib = sys.modules.get('heartlib')

'''
Remove a namespace-only import without discarding a valid package.
僅移除錯誤的 namespace 快取，保留已正確匯入的套件。
'''
if cached_heartlib is not None and getattr(cached_heartlib, '__file__', None) is None:
    del sys.modules['heartlib']

from heartlib import HeartMuLaGenPipeline
import heartlib
assert Path(heartlib.__file__).resolve().is_relative_to(REPO / 'src'), f'heartlib 匯入來源錯誤：{heartlib.__file__}'

'''
Assign token generation and audio decoding to separate GPUs.
將 token 生成與音訊解碼分配到不同 GPU。
'''
devices = {'mula': torch.device(f'cuda:{MODEL_GPU_ID}'), 'codec': torch.device(f'cuda:{AUX_GPU_ID}')}
if globals().get('pipe') is None:
    pipe = HeartMuLaGenPipeline.from_pretrained(
        str(CKPT), device=devices,
        dtype={'mula': torch.float16, 'codec': torch.float32},
        version='3B', lazy_load=False,
    )
    print('已首次載入模型；後續生成請執行第 4、5、6 節。')
else:
    '''
    Confirm that the existing pipeline is the intended implementation.
    確認既有 Pipeline 是正確的官方類別。
    '''
    assert isinstance(pipe, HeartMuLaGenPipeline), '既有 pipe 不是 HeartMuLaGenPipeline。'
    print('沿用既有模型，未再次載入權重。')

assert not pipe.lazy_load and pipe._mula is not None and pipe._codec is not None, '雙 GPU 模型未完整載入。'

'''
Verify that all HeartMuLa parameters are on GPU 0.
確認 確認 HeartMuLa 參數均位於指定的生成 GPU。
'''
assert {p.device for p in pipe.mula.parameters()} == {devices['mula']}, 'HeartMuLa 並未完整位於 指定的生成 GPU。'
assert {p.device for p in pipe.codec.parameters()} == {devices['codec']}, 'HeartCodec 並未完整位於 指定的 Codec GPU。'
for gpu in sorted({MODEL_GPU_ID, AUX_GPU_ID}):
    print(f'GPU {gpu}：{torch.cuda.memory_allocated(gpu) / 1024**3:.2f} GiB 已分配')

print('✓ 第 4 節完成：音樂模型已載入並完成 GPU 分配設定檢查。')


5. 生成設定


In [ ]:
'''
Define the lyrics for the next generation.
設定下一首音樂使用的歌詞。
'''
LYRICS = '''
[Verse]
十年生死兩茫茫
不思量，卻自難忘
千里孤墳隔月光
無處話淒涼

[Chorus]
夜來幽夢，忽然還鄉
小軒窗前，你正梳妝
相顧無言，淚落千行
一醒只剩，滿地月光

[Bridge]
料得年年最斷腸
明月夜，短松岡

[Chorus]
隔著十年，隔著陰陽
明月仍照，那座松岡
'''.strip()
TAGS = '''
soft,sad,ballad,longing
'''.strip()
DURATION = 20

'''
Choose a fixed seed, or -1 for a random seed on each generation.
設定固定種子，或使用 -1 讓每次生成採用隨機種子。
'''
SEED = 42
OUTPUT_NAME = '十年生死兩茫茫.mp3'
print('✓ 第 5 節完成：歌詞、曲風和生成參數已設定（尚未生成）。')


6. 生成音樂（可重複執行）


In [ ]:
'''
Serialize notebook generation with optional remote FIFO GPU jobs.
讓 Notebook 手動生成與選用的遠端 FIFO 共用 GPU 鎖，避免同時推論。
不必先啟動 ngrok 或任何 HTTP API。
'''
import threading
GENERATION_LOCK = globals().setdefault('GENERATION_LOCK', threading.Lock())
with GENERATION_LOCK:
    import time
    import heartlib.pipelines.music_generation as music_generation
    import random
    import secrets
    import numpy as np
    RUN_SEED = secrets.randbelow(2**32) if SEED == -1 else int(SEED)
    assert 0 <= RUN_SEED < 2**32, 'SEED 必須介於 0 至 4294967295，或設為 -1。'
    random.seed(RUN_SEED)

    '''
    Seed NumPy before sampling.
    取樣前同步設定 NumPy 種子。
    '''
    np.random.seed(RUN_SEED)
    torch.manual_seed(RUN_SEED)
    torch.cuda.manual_seed_all(RUN_SEED)
    print(f'本次生成 seed：{RUN_SEED}')
    lyrics_file = ROOT / 'heartmula_lyrics.txt'

    '''
    Update the input file on every new generation.
    每次生成時更新歌詞檔，確保使用最新內容。
    '''
    lyrics_file.write_text(LYRICS, encoding='utf-8')
    tags_file = ROOT / 'heartmula_tags.txt'
    tags_file.write_text(TAGS, encoding='utf-8')
    filename = Path(OUTPUT_NAME)

    '''
    Reject directory paths and non-MP3 extensions.
    防止輸出到其他資料夾，並限制副檔名為 MP3。
    '''
    assert filename.name == OUTPUT_NAME and filename.suffix.lower() == '.mp3', 'OUTPUT_NAME 必須是單純的 .mp3 檔名。'
    output = ROOT / filename
    index = 2
    while output.exists():
        output = ROOT / f'{filename.stem}_{index:03d}.mp3'
        index += 1
    original_tqdm = music_generation.tqdm

    '''
    Print actual generated-frame progress, even without Notebook widgets.
    即使 Notebook 沒有啟用 widget，也能顯示實際逐幀生成進度。
    '''
    def show_progress(frame_indices):
        '''
        Read the total number of 80 ms frames allowed in this generation.
        取得本次允許生成的 80 毫秒音訊幀數量。
        '''
        total = len(frame_indices)
        interval = max(1, total // 10)
        print(f'音訊生成中：最多 {total} 幀（{DURATION} 秒）', flush=True)
        for frame in frame_indices:
            if frame > 0 and frame % interval == 0:
                print(f'長度進度 {frame / total:.0%}｜約 {frame * 0.08:.1f}/{DURATION} 秒｜耗時 {time.perf_counter() - start:.0f} 秒', flush=True)
            yield frame
        print('長度進度 100%｜已達音訊長度上限', flush=True)
    used_gpus = set()

    '''
    Capture the device of each selected layer's output tensor.
    以指定模型層的輸出 Tensor 裝置確認實際運算位置。
    '''
    def record_gpu(module, inputs, result):
        '''
        Identify the device used by this model operation.
        取得本次模型運算實際使用的 GPU 編號。
        '''
        gpu = result.device.index
        if gpu == 1 and gpu not in used_gpus:
            print('[2/3] GPU 1 正在解碼音訊...', flush=True)
        used_gpus.add(gpu)
    mula_hook = pipe.mula.codebook0_head.register_forward_hook(record_gpu)
    codec_hook = pipe.codec.decoder.scalar_model.decoder[0].register_forward_hook(record_gpu)
    start = time.perf_counter()
    print(f'[1/3] 正在生成 {output.name}，音訊長度上限 {DURATION} 秒...', flush=True)

    '''
    Use text progress for this run only, without modifying model weights.
    只在本次執行使用文字進度顯示，不修改模型權重。
    '''
    music_generation.tqdm = show_progress
    try:
        with torch.no_grad():
            pipe(
                {'lyrics': str(lyrics_file), 'tags': str(tags_file)},
                save_path=str(output),
                max_audio_length_ms=DURATION * 1000,
                temperature=1.0, topk=50, cfg_scale=1.5,
            )
    finally:
        music_generation.tqdm = original_tqdm

        '''
        Remove the GPU 0 monitoring hook after this run.
        本次執行結束後移除 GPU 0 的監測 hook。
        '''
        mula_hook.remove()
        codec_hook.remove()
    print('[3/3] 已完成音訊生成與儲存，正在檢查結果...', flush=True)
    assert used_gpus == {0, 1}, f'雙 GPU 運算驗證失敗：{used_gpus}'

    '''
    Require the generated MP3 to exist and contain nonempty audio data.
    確認本次成功產生非空的 MP3 音訊檔案。
    '''
    assert output.is_file() and output.stat().st_size > 1024, '找不到有效的 MP3。'
    for gpu in sorted({MODEL_GPU_ID, AUX_GPU_ID}):
        print(f'GPU {gpu}：已驗證運算；VRAM {torch.cuda.memory_allocated(gpu) / 1024**3:.2f} GiB')
    print(f'MP3：{output}；生成耗時：{(time.perf_counter() - start) / 60:.2f} 分鐘')

print('✓ 第 6 節完成：MP3 已生成並通過檔案檢查。')


7. 試聽與下載


In [ ]:
'''
Import notebook audio playback and download links.
匯入 Notebook 音訊播放與下載連結工具。
'''
from IPython.display import Audio, FileLink, display
display(Audio(filename=str(output)))
display(FileLink(str(output)))
print('✓ 第 7 節完成：試聽和下載連結已顯示。')


8. （選用）啟動 ngrok 與 FIFO 遠端 API


In [ ]:
'''
Require the first configuration cell before enabling the optional remote API.
選用遠端 API 啟動前，必須先執行第一格共用參數設定。
'''
assert all(name in globals() for name in ('REMOTE_PORT', 'REMOTE_SERVICE_ID', 'REMOTE_ROOT')), '請先執行第 1 節共用參數設定。'
import zlib
import math
import re

import queue
import copy
import secrets
import io
import json
import os
import shutil
import subprocess
import tarfile
import tempfile
import threading
import time
import urllib.request
import uuid
import zipfile
from email import policy
from email.parser import BytesParser
from getpass import getpass
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from pathlib import Path
from urllib.parse import urlsplit, parse_qs

REMOTE_KIND = 'music'
REMOTE_FIELDS = ['task', 'lyrics', 'tags', 'duration', 'seed']
REMOTE_IS_LTX = False
REMOTE_SCRIPT = zlib.decompress(b'x\xda\xadZ]S\x1bY\x92}\xe7W\xd4K\x87\x84\x17d\xc0\xdd\xb3\xdd\xdea#\xd8\xb6\xa3\xc3\xb1c\x9bp\xdb\x13\xbb\xd11\xa1(\xa4+\xa9\xb6\xa5*M\x95\x04f\t"\xc0m\x1a0`\x88i\xc0\xe6\xc3\xc6\xc6_\xf2\x07`\xbb\xb1\x8d1\x1f?fT%\xe9\xc9\x7faO\xde\xbcU*\t\xf0\xcc\xc3\xfa\xc5\xa8T\xb7\xee\xc9\xbc\x99yNf)\x12\x89\xb4\\\xc8\xe5-\xbb\xa0\x152B\xcb\x1a}\xb6n\x1b\xc2\xd1L!\x92"\xa9\xa5,\x1b_\x18\x8e\xe6\x88D\xc1\xb0\xccX\x8b;\xbd\xe5\x8e=\xf1V_y\x9b\xef*\xa5)wo\xd6\x9b\x1c\xa9\xad\x8eT\x96o\xba\xe3\xfb\xf8\xe8\xee\xbe\xfc\xfb\xc8\x8d\x96\x08\x9el\xa8\'\x1b9\xe1\xff\x9d\x11\xba]\xc06\xb1\xbc\x91\x17Y\xc3\x14N,Wt\x8cD<-La\xeb\xb4\x87\xa6;Z\xf35\x7f\xb9\xad\x9bI+\xe7\x7f\x02([\x14\x1c\xff\xa3Y\xcc\xe5\x07i\xb5\x99o\x91\xfb_\x11\x8e\x95\xed\x17\x9a\xae\xd9"o[\xc9b\xc2\xe8\xcb\n,\x13\xc9\x7f\xd3\xda;5\xf5|\x98\xab\xc3\xe0\x01\xf9E\xddd\xbb\x08s\xcb\xfb\x87\x95\xf9\x92\xbb\xb2\xebn.WJ\x9b\xee\xc6\\m|\xa62{Py7\xe7\xdd_\xfd\xbc\xb7\x82\xc7\xb8\x93\xe3\x95\x1b\xbb\xe4\x92W\x0f+\xf3k\xde\xc4\\e~\x9d\xfeX|\xcdK|\x7f\\\xb9v)\xfe\xe3\xf9\xf3\xe7\xb4n\x1fz\x8c\x0c\xea\x13Yk \xdau\xea\xd4\x99\xaeV\xcdHi|K7!\x14YGh\x86Y\x88\xd2\xb5V\xb6\xea?\x85\xc8\xcb\xd3\x92p\x07\x0c\x805\xf9sF\xb7q\xe5LW{\x9f!]\x95\x16\x9a\x95\xd2.\x15s\xbdp\x8b\x99\xd4z\x07\xafZv"\x13kQ\x96,\xcd\xb9\x13\xef\xdd\xd5\x92\xba\xa5:1\xee\xdf\xa2\xb9\xf7\xc6\xbd\xf9-ov\x1b\xe7\xea?\xb1\xb2u\xd3]\x9d\xf1\x8d\xd1\x1dG\xc0\xe9\x1d\xda\x1f\xbb\xb5\xc0\xb0?j\xd2\x8c6-"?\xba\x87c\xb5\x07\x13\xe5OS\xde\xe2\x1e\xee\xac\x8e\xff\xae}\xdd\xf5\xdd\xd7\xdf\xfd\xe1_\xbb\xbe\xfb\xe6\xf3\xde\xb47\xb1X-m\xc0y\xb0\x15\xcf\x8d\xb0\x81?\x92]z6\x1b\n\xc7>\x81c\x81\x85z.\x8f\xa0IK\xab\xad"\x8c\x14}E#\x9b\xa4K2~-\x9d\xa26g%\xe1\xb8X\x8bw\xbbT}\xb9\xe8\xce.z\xa5G\xee\xe4\x0c\xb6\xc2)\xd6\x96K\xde\xf35\xf6\x00 \x94wfp\xa28*\xf7\xd3n\xe5\xe5\x94\xfb\xe1muo\x12\x11\x0e\xbb\xbd\xd2C\xf7\xfe\x94o/\x87^\x8c\x9c\x1e\xf5\xedm\r\x01f\x1f6\x01E\xbe\xf8\xbb\xbbs\xd3\xde\xc6\x13\xc6\xa0nn\x8c\x0e3\x1f\xfbG{\x90\x8d9\x1d\xc7\xed\x1f\x13/\xf0\x03\xd9\xb2c-j\x03\xff\x06\x98Q\xde\xf9T}:\xcaf{\x0b;\x1c\x9b\xeeR\xa9q\xf7\x82\x8c\x8c\x9cn\x16\xf5l\xfcD\x00\xcd\xfb9\x1a\xe5+\x8eJ\xef\xd7\x8d\xacN\xd9\xf5C\xef5\xb8>l-\n\x84\xb7:\xe9\xcen!\x93\xe8k\x02\xf5\xcf\xc1I\x14\x93z\x18S\x1c[\x1d\xc1\x95\xb1\x06\xa4cD*EE\xaa_\xd4\xf3\x18i/\xf4\x82\x84%+J{\xa8\xca \xed\x91\x7f-\xb5\x87[\x95\xc7\xbb\xee\xd6\xd3\xda\xf2J\x10\x14\xde\xe2\xc7\xf2\xc1\xa1\xbb\x01tk\x9c\xef\xb0\xa7\xbc3Ryw\xb3\xb6\xf6\xbbW\xba\xc1i\xce\xe6\xf9\x90\xf36%j*\x12.\x04\x12\xca\xe7\xbd\xe5!\x1f\xf4p\xc4\x87\xad\x03\xa8\x8cY*@\x05-;h\x1b\t\x80\xba\x8ezi\xa1\x1e\xa5\x0c\x80\xd6\x13\t\x91/\xc0\x98\xbeAy\xaf\x95J\x19\tC\xcfj~\xf5\x84\xa3_\x8f{\xab#\x88_oc\xba\xfa\xfc\xbe\xbb\xf5\x12\xc1\xebn\xde\x85\tZ\xaf\xbaK\x83\xef\xab\x9b#\x08E\x8a\xea\xc5qw\xe3\x8e\xf7b\xde\xc7-wv\xe2rC\xa4\xf2\xe5\xcbW\xb5\xd3ZD\x16\xea\\1\xab\xc7\xf9\xfbX\xe1zA\xe5\xe7\xb5|\x12\x98% \xc3\xcc#\x0b\xe5R\xb8T\xf4\x0b{PV\xd2\xba\x9f\x91\x85[\xb3\x81?\xbc\xa5\x1b\xde\xcav\x80\x16(\xe0\xed\xca\xfan\xf9\xf0\x1eWZ\xb6\xc5\x1d{\xe6n~<\x06`l\xc06\n"Nn\x8a\xfe\xe9\xbf\xaf\\\xf8\xfe\xc76M\x98\t\x8b*@w\xa4XH\xb5\x7f{\x92\x83\x9d\xc2 P&\x85\x93\xb0\x8d<G\x80\xa5\x19\x05\'dC\xd8\x9b+ok\xeb%\xaf\xb4Ty\xf3\x98}Z\xdd\xdb!\xe2{1\xef=\x9c\x08\xa2TO\x7f\xc1q\xf4\xedQ\xb71\x0e\xfa\xce/\x16BG\xa2\x9e\xe4\xb1\xf2\xc7)\x14\x0f\xe5\xb4\x10\xa8#\x10\xc2\xae\xb9\xda\xf3\xc3\xc9\x8e\xb9`\x16\x84\x9d\x07\xfbH\xef\xd0RS\xcf\t-a\x99)#]$\x06\xd1\xfb,bM\xa2\xc5\xac\x95@\xb8]\xec=\x13\xdc(\x9d\xa4\x12\xfb\xc1\x9e7=N\xcc\x88\xb0z1\xef\xce\xcdT\x9f.**|\xbe&W\xf1e\x1fl\xb0Y\xb7\xd6\xab\x172\xd1\xcb\xd7\xae\xf6^\xbb\x1a\xbf\xd4s\xf1|\xabO\xd9\xff\x83$\xd6\x92\x86\x8d\xff,DS\x1e\xf79\x92\xbdL\xcbl\xa7G\xc2@a:p\x14e\xef\xdd\xb7\xde\xc6:\x1d\xcd\xf8\xae;\x81\xb8y_\xfe\xb4X\xfd}\xdc[\\r\x1f\x1f\xc8\xf2\xfeT\xd1\xdc\xe4\x16c!\xb6\xc1c\x9ah,\xb0\x8e\xe1uk!hr\xf7\xe0\x06\xa7\x88$\xbc\x1e\x03e\x0b;\xdaJ\xb7Fb\xb9\xfc\x99\x088/\xbc\x86\xa9\xcf\xbb\xbb\xe5.nV\xb6\xe7\x89F\xe9\xb6\xbaC\xfc\xa8p8JC\xee\x0f\xbc\xc4R\x84\xce\xc8F\x00\x87#\xa4\xf2\xea\x15\xcaQ]o <n?Q\x19\xd4x"\xbe\x9d`K\n\xf3 N\xfdMT\xb2\x14t\x16P}\xc2&2\xd5A\xec\x1d]\xc8\xe9\xec\xa06\x90\x11\xac/pOZ\xa8l\xd7\xb3\xb6\xd0\x93\x838\x0cC\x96Qw\xf6\x05j<\xef\xcd9\x02&u7\xeeB] \xe7\xa9\xa4\x82\x01\x0f\xee\xcb\xa7\xd6\x16\xa7\xdcgS\xee\xad\x07\xe5\x9d[\xde\xbbQ\xef\xf5vui-\xd0\x8cfR\\\x07\xcc.\x06\xd6k\x0b\x9c\x0f\xa2\x11\x01\xdboXE%\r\xa9 \xeaI\xc9\xfb\xba\xc6\'B\xe9\xcc\xf5>\xe4Bf"\xb8\xe1(\xa6\xea\xf8\x0bwj\x81QT>\x94\x00\x01(k\xa3\x87\xee\x18\xa2\xf8\xd7\xeaoS\xee\xd8\x042\x8f\xb8J\x96|\x1f\xe0@FV;\xe9\xce\x18\x9b\x1fm=\xdb\xd2\xa2\xe1\x1f}O\xff\x7f\x9f\xb1,u\xae&U\xf4:;\xb2\x8b\x81\xb01\xa5hQ\xf9\xe0\x9e\xbb;\xeb\xbe\x9e\xf2&\x0f\xca;S8_wd\xca}\xbd\xec\xad\xbe\xe0\x93\xa5\x18j\xcc\xa9\xf0\x9eG\x0e82T\x8f\xd9\x82\xc8\r\xc7\x87\xa4o\xcfv\x9cI\x0e\xcb\x88m\xc4|\x91r\x1e>\x0c@3T\x12\xa4\xbai\xe1\xaa\xcd\'\x9fA]\x90j\x93"\xb4\x8e\xbe:\xf5\xa4\xfa\xfc-;\xb2\xfcq\r\x9ef\xd0T\xdf?\xeeUv\xa7\x9b\x0c\xe3;\x9b\x8d\xe0\xd3\xff\x97n\xad\xb3I\xeb\x16\xf3N\x01\x11\x97C\x1cXi\xc4\x04\xca6\x14\x96\xc8\t\xb3\xc0t\xcet\xef\xa0f\xa8&"\x85*\x17\x12\xf2\xde\xf2K:zI\x8c\xb5\x91\xb7\xee\xeeS\x94(\xd4\x05\xc9\xf6{\xcc\xd9\xee\xda\x87\xea\xc3i\x92\xf6\xf7\xd6\xdc\x83\xe9\xda\x8d\x9b\xee\xed 0-\xdbH\x1b&\x84H\xe1\xaf\xc9\x1c\xdc\xdc\xdc\xa2\xc4\xe8\xba\x1f\xb4P\x02 \xef\x02tK\xd0d$\xdbS6\xb9\xcc7\xa0\x8d\xe8\xd2\x0c\x94\xec%\xab \xfa,\xebg\\H\xa6\x85L\xa9\x99\xdf\xe1\xc2\xfa\x17\xde\xdb\xbf\x91\x8cZX#\x19\xc5wQ\x81\xfb\xb8V\xfde?\xacbj#s\xee\xc7\x11.\x0cl\xa9oBR\xa4\xd0$X\x03q\x1fCTB\x8a\xc3\xebFB8\xcdQ|\x05\x19\xce\x89o\x15`\x88\x8a\x07t\x14\xdfvh9G\x93k\x1dR\x7f(\x85IM6!\xf0v\xb8L\xd1S\xa0:\xdc\x83;\xca\xc1c\xa3\xd5\xd2\x1bU\xb3\x10\xcdx\x90\xb7\xf5\xb2\xf2\xecoH\xb1j\xe9\x16`C\x11\xd6\xc6g\x9b\xc3\x82\x01tk\x08\xe7&\xc8\xcd\x88\xb9K\xb4\x8a\xe9\x0c*\x17\xeb\x91\xce\x8e\xaf\xa4\xa4\xea\xb7\x8c$\xea\x15V9$\x12\xb3VZ\xe3\x06\x94a>~V\xd9\xbe\t\xe6\x95\xf7\xbb+\xf7\xdd\x07o\xb8\xc4\xd6\xcb\x82\xa4\x99\xda\xe8m\xf7\xf1re{\xa46}\xf3h\xf8"\xe8\xfa%\xd4\x9c~=\xda\xd9\xa6\x90\x9f>\x8d\xa76a\xbd@\x16\xb0\x8eB\xb5\x95V\xb5\xf7\r\xaa \t\xe9T\xca\xb7>\x91.*w\xaa\x93\xfe\xf0\x96\xab\xa8\x7f\xd8$\xa2\xd9\x89\xcd\x98|Y\xca\xdf*Q\xb1\xb3\x01Q\n\xa5\x03K\xb4!\x89qX\xc3s>\xefM\x0c\x9d\xbbv\xa5\xe7\xea\x85\xcb\x97\x865\x9c\xcb\xe7\xbdI\x10[*[t2\xddW\xed\xa2h\xb2\xa1\x17\xec\xc9\xf2\xc5\xcf\x0e6D\xe5\xb1\xaa&\x81j\rY\x95\xb5\xac\xbcr<$\xd7\xd6,\x95\xba\xdbkD\x0b\xb0f{\xdd\xdd[(\xef>\xa6S\x90\t\x8bXQ\x8a\xfbp\xdb]\x9dh6\x91r\xdf\xdfWk\x88\x0f\x15\xd1\xe1\x9bCq\x92\xb0\xa8\x86H\xbeP\xb1L\x1de\x1a\xda\xcd\x0e\x8eR\x85\x87D\nj[]\xe3\xd0 &\xd9\x9c\xe60>!x\x9b7E!e\x90\xff\x8eVXJ\n\xf9\xe9\xabP\xd4tk\x1d!\xc4\xcd\x0f\xa0\x7fA\xb7S/+\xc9\xa2\xefS#g\xc8HJp\x05\xc4\x16\x03\xc8\xce\xf6\x04D\xdc\xcfr\n\x13kx\x16\x87\x92*\x14\x0b\x1fP(@\x85PKd\x92\xac\x1bT^>=qgo\xa9\xf6\xe8\xfdo\x08~Pgm\xf1\xb7\xb0\x91\xc7\xe1\x0c\xa2N>\x97\x1f\xa7\r\xb1\xc5\xa79+\xce\xc6:\xbe\x1a\xfe\xbc\xbf\x8a\xb4\xf3\xbf9\xa5u\xc4:\xbe=\x1b\xebL\r\x9fn\n\xc4\xfd\xd5\xea\xc8\x1d\xec\x8dp%C\xf2\xc2N\xc5\x13V\x91\\\x07\x15\xd6\x0eY\r\x05\x83G\xa6\xe4\xed\xc7\x05m3L\xc9-\x86\x99\x02\x1d\x9b\t\x81\x1c\xcb\xe8P\x18\xb6V4\x13\x19\x9a\x9d$\xeb\xdeB{\xe2M\x8fr,\xf2P\x81x\xe2\xc6nyg\xa6\xbay\xeb\xa4\x03\x1f4DV\x9d\xf2\xb1e\xaa\xb3\x03\x95FJ,\x84\x86\xd4yEt\xd2a!(\x9d\xc2\xc7J<F\x99\xe6\xc3r\x7f\x19\x83\x8e\xa1\x18\\\xd8\xae\x8d\xceC\xfc\xb2&o\x8aG>Q:3y\xd8r\xcf\xe3+D\xe3Q\xd1}\xf09\xd5\x99\xd1y~`8D\x9a\xfd\xcb\xd2=a\xd9\x08\xb9\x8c\x81z\xf0\xfd\xb5s=h\xb6\xfa)\t\x15\x13RA\xbe.\x12E\xd5::"\x0b\x89\xefOk\xd0\xa5\r\n\xdb\xa1\xf9\xc5]\x14VE\x172\xec\x98\x95\x95\xb4\x94#\x19\xf7\xcdcb\x0f\xb9G\xf5\xd1\xbd\xca\xfe\xa6\xcfpEG$\xe3\xe9<d"\x8d\xd8\nQ\x05\xed{=_\x80G\xe5\xbe\x0c\x8ahL\x16\xae\x00\x86\x04\x10q|\x1dE=\x06\xcdS\x90\x00G\xb7f&\xd0\xae\xca{\x14\x04t\xb1\xd5\x173\x8a\x82G\xa7*\x9bw\xca\xfb3!hD\xbe\xb6t\x11\x01\x8c\xc2\xeabV\xb4q\xf7\t1\x00:.f\x0b\xcd\x14|!\t\x81c\xa4\x06\xc3\xc8\xc9F\x1e\x08 &\xd8yV\xfeD\xcae\xd8\x0c\x88\xc1\xd5\xc5\xa4\x9c\xc5\x1c\xab\xc3\x80\x10\x1edL1\xde7&kz#\xbc\x1e\xd3D\n\x02\x93^L\x1a\x16\x00r\xc3Yo\x19h\x87N\xbe.\xf5#\xf51\x90c\xaa\xa8V\x16\xde\xab;\xc8\xa7\xcf\x1eU\xd6\xf7\xdc\xa5R\xed\xe9b\xa0\xc5\x82\xc0\xe5o+S#\xde\xd8\xd4\x11\xd6M1^\xa8FY\xf4\xe8\x03\x14+\x91A\x10\x0f\'\x10\xc1\x8f,*\xd9\xbb\x0c\xbe\x1d\x95$-4\xd0\x81C\xff#5,vuBd\xb3!2@\x13\x01\x07?\xd8S\xcd\xe7\xcc\x08\xf4Z\x00\x973\x86A\xe3\x0fot\xeb\xa4"\xa1\x92\xef\xa7\xae\xd3g\xfe\xa2\x9c\xe1m<\xc2\xd3y1?(\x16\x8b}\x89\x86{\x12\x89"M\x1c\n\xecp\xc5}\x9a\xd5\'[\xa7\xa4\x06\x86\xe0\t\xe91\xf2,\x9co\x8a\nFo\xd3`?\x94{\x1c=_\x8e\x98\xc0\xd31\xf4eQ\xfc\xa1R\xef2\x83@\xc9\xffY\x98\xe1\xf9\x1b\x91o\x91\xc5\xbb\x9c"\xd2\xa3;\xd0\xd9\xae<\xf2v^\xf1\'\x8dZ3\xec*\xd7j>:\xc2\xe2\'\x95\x9c\xb3dH\x1dw\xcb\x99X\x8c.\xc4(\xdaH2w\xc43\x90\xb01P:\xba4a\xc7\xa1\x13\x06t\xe4\x1f-\x88\xd6s\xb1\t\xe9\x80\xdeOs\x99\\\xbb\x1f\xb6\xc7!\xedl@\xda\xa9\x90\x86\x8f\xbd\x11(=)\xd1\x80T^\x89\xa9=bNB\x87\xea\x88\xcbt\xf6/\xfe\xd4\xf1\x97\x7f\x16;\xf7\xef`F\x7f\x12\x9e(\xda\xa0\xb5\xf0\xd0\x80\xdb V\x8d\xd5\xd2\x04\xb0\xf1\xa1\x87\x07\x9a\xde\xe4H\xf5\xd6\x9b\xea\xdbO4>\x08x\x9e6\x90\xf4\n\xe8\xc7\x90\xaf_\xfd\x83\xb7H4\x00TUT\x8e1\xd4\x98\x8b\x96\xb6\x83\xdd\x9c\xa2D\xc9\x95+\xe0_\xf4;\x88yI\xf1\xc1\xe8]\xcd\xef\'g D\xd0\x8dsj\x85Qs\xee5\x0e:|\xe5\xf1S\'e\x14\xe7\x92\x1a\xc6\x0e\xa9\xae\x9dP\r\x93\xac?\xc2kZ\xa3\xea8&\xef\x82\x99\r\xb5\xc8A3\x1a~i$)\xbd-h\xec`\'\nx\xdd\xe2\x01a\xa43jd\xa2\x8a\x88_\xed\xd4\x00G\x0eh\x99\x87\xd9d~MQ>\xdc\xf4\xe6?*\xcf\xbc\xbaU\x1b\x9f\xa9g\xc11\xad(1`\xb8\xd7c\xe8\xe7\xe1~\x9cE\xce2\r\xb4\xca\x84\x8a\x02\x8a\x07l\xf5\xe9\xb2o\x96\x8e[\xb9\xa9F\x15\xa1&[7\xb2X\xee\xf7\xa6\xea|\x1e\xbf\xf1\x16\xee\xa8V\xf4\xe9(w\xcdG\x1b\xed\xf2\xce\xd3\xca\xb3O\xb5\xa5\xc7*oh\xdf`\x88i\x0f6\x11\xdf9\xc3\x91\xf3\x92\xb4\xad\'\r\x04\xb2So\xe1\x8a\xa6)\xa8\x89\xd3\xd1\xdb\xd5\xe5[N\xe4h\\\x882\xa48pt\x95|\xb9\xbe\x05\x00\x1c\xeeA\x1f\xc7\xb0)\xd0d\xdb\x04\xd7\xba\x87c@N$T\xba\xeb\xfd\xfa\xbe\xf6r\x1ek\x8f\xe9C\xe9L5~5aZq\xc2\x16\x8c~\x8e\xb6\x18\x80\xa2\xf5YX\x00\x0f\x1a\xc4\xe3\xea\xbd\x14\x02E6\xa8\xc7O\xf4#\xe1\x12\xad\xf5\xf4^\xa8SNmd\x96\x9a\xcf\xa6\xd9>\xbd\x9f\x98/\x91P{6WYX\xa2*\xb4\xb2$\xab\x92R,\xdc\xa6p\xfb41\xce\xc5\xe9D6\xc2S\xa3\rB~(\xc2\xe3\xf7\xc8YHl;\x1a\x9a\xc5\xb7\xb6i\x11\x9a>\xabo\x82At\xebp[\xc3\x13\x1c\xd4\xd38\x8dq\xbb\xe96N\xc1\xd6\xc6[\xd0,\xc7\xa5\x84\x88\xa3\xc3O\x172\xf1\x9c\xd3\xed\'"\x9a\x02\x88\xd1\x8e\xc6\x05\x05\x91\x93\x92\x07\xc1\xd8\xdd\x19\xeb\xa0>;\xffs\xf77\xf8#\x91J\xc7\xa9\x98\xd2\xf5o\xea\x8bZ[R\xd4\x9ef\x07\x8f\xcc8dt7\x9dE\x90\xd6\x107\xc1t\x89g`\xfe9Q\xa4)\x19s4\xe2\xf9M8\xa7\xad\xbb\xff\xa6\xb6v\xc8\x93\xe9\xfa\xa9\xa9T\x87\x88_\xff\xa59\xcaNJ\xe7\x86\tT\xb3\x199K\xbd\xee`\xe6lJ\xf0#\xe30Zt\xec\xd0\x8bsT=\x85\xfa\xee\xc6lm\xc4\xa9\xc8\x17,E\xdbG[\xbf\x08\xaa\xf3\xff\x05T\xe7?\x00U\'\xda\x10*\xd9\x03P_e\xe7x\xdc\x12\xfe\xb5\x81\xc9\xc3X\x84)\x01\xb3\x05\x82\xca\x94\xcd\x97\xc5\xc2\xb0\xfe\xae\x8d%~\xe8\xb5Z0\x04\xf0\xb3K>\xc9\xbdISf\xef\xdd(dT#+E~:C\xa4\x14\xac\x0b\xcfd\xb0\x9a\x17\xd2DR\xb2\x96\xf7b\xdd[{\xc2\xbf48\x81\x89\xfe,l\xee\x0et\xa6^Ua\x82\x86\x8b\x98\x1aR\x86\xaa\xb7\xdf"H\t#\xeb\x12\xbf\xa9U}\xcb\xd8sw\xef\x9d\xf4p\xed\x97\xfd\xb0\xf8+\xef\xfe\x1a\xee!\x9a\xde\x9b\x84\x1a\xaenm\x08\xf9\xd79\x0c\x98\x11\xbf\x04\xf1\x9a\xda\xf3;\xd5\x8dI\x9f\'\x96\x87\x82E\xc3\x11_<\xfc\xb5h\xa8,\x0c\x06\x1b\xd2\x97\xa8\xfar\xc6.\x8f\t\xe2\xa1@o\xbfM\xcbD\xfe\x17\x06\xfd\xb6C/\xe8\xbe\x1d\xaa\xe9\x99\x98so\xad\xa9\xa9\xd8\xbd\xfb\x95\xe7\xbb\xfe\xec\x9c=\xde\xf8\x16O\x99\xa2\xc4\x81\xc1U,\xda\xca\x9c\xc8\x17!\x7f\xd0L\xe2\xbf\xb8c\xfc/Mp:;\xba\xbeF\xfd\x93\x13\xed\x194\xdf\xf4\xf2caB\xedR\x7f\xbb\xa3T\x91\x1a\x05+\x8e\xa2\x89i\xe2\x98\xb3P\xf9\x10V\xe8<l\n\xabsdC\xe8\xac\xe8W7\xf2\xac\x02\xe2r\'~\xad\x8d\xd5_\xb8\xa1nQ;\x04\xa7\xa1S\x85S\xa3C\x17/\x9f;\xff\xa78\x16\xc7/\x9ck\xd3z\xae\xfd\x97\xfa{\xb8\xb9\xf7T\xa3\xa6pd!;\xe8\xe7\x07N\x9d\xd2\xfcWT\xf2\x8dAso\xa1\x04\x9b\x8c\x9e\xf2\xa7\x19&(\xfa\xbd\tv\xa4\xe2x\xb0\xc2\xaf\x03\x8e}\x8fu\xd2\x1c\x93\xcc\x1e\x82M\x10p\xcb\xc4z2\xb68\xce>\xef\xad\xfc\xf9J\xcfE\x9ah\xd6\x7f= }\x1eW>\x17\xdc\x9ah\xa7\xe5\x01\x9e:u\xe6l\xac+5\xac\xfd`\xfcG\xa4Y\xc7rQ\xaa\xd3\xb1z\xab$\xa3Bd\xf5<u\xe1\x10\xb2\xc5\x82\x08~E\xd0\xf0\x13 \xf9\x02\x8e\xde9\xcd\xcdTJo(\xbb\xf9\xa0\xa4\xc4m\x16\xab\x08\x1aJ\x0c\x8e6X\xb6\xa2j\x82\xbc\x99\xbe\x89~a\xeaE\xe6\xfc\xa1\x83-\xa1\xe3\x9f\xbb\x0bc\xfe\x0f\xae\xaa\x86\xa7').decode('utf-8')
REMOTE_ALLOWED_TASKS = ['music']
REMOTE_MAX_IMAGES = 0

REMOTE_ROOT.mkdir(parents=True, exist_ok=True)

'''
Share existing state when the API cell is executed again.
重跑 API Cell 時保留原有工作及鎖定狀態。
'''
REMOTE_JOBS = globals().setdefault('REMOTE_JOBS', {})
REMOTE_JOB_LOCK = globals().setdefault('REMOTE_JOB_LOCK', threading.Lock())
REMOTE_ENGINE_LOCK = globals().setdefault('GENERATION_LOCK', threading.Lock())
REMOTE_QUEUE = globals().setdefault('REMOTE_QUEUE', queue.Queue(maxsize=REMOTE_QUEUE_CAPACITY))
assert REMOTE_QUEUE.maxsize == REMOTE_QUEUE_CAPACITY, 'FIFO 容量已變更，請先重新啟動 Kernel 及遠端服務。'
REMOTE_UPLOAD_SLOTS = globals().setdefault('REMOTE_UPLOAD_SLOTS', threading.BoundedSemaphore(REMOTE_UPLOAD_CONCURRENCY))
REMOTE_RESULT_TTL = REMOTE_RESULT_TTL_HOURS * 3600
REMOTE_RESULT_BYTES = REMOTE_RESULT_LIMIT_GIB * 1024**3
REMOTE_MIN_FREE_BYTES = REMOTE_MIN_FREE_GIB * 1024**3
REMOTE_STALL_SECONDS = REMOTE_STALL_HOURS * 3600  # 無限大代表不因等待時間長而拒絕工作

'''
Verify the CUDA backend and existing model without loading another model.
確認 CUDA 與已啟動的模型後端，不另外載入第二份模型。
'''
def remote_gpu_status():
    try:
        import torch
        if not torch.cuda.is_available():
            return False, 'CUDA GPU 不可用；請檢查 NVIDIA 驅動與 PyTorch，並重新執行模型初始化 Cell。', []
        qwen = REMOTE_KIND == 'image_multi'
        yue = REMOTE_SERVICE_ID.startswith('06_YuE2_')
        if yue:
            selected_gpus = [globals().get('GPU_ID', 0)]
        elif qwen and globals().get('QWEN21_STATE', {}).get('mode') != 'bf16_dual':
            selected_gpus = [globals().get('GPU_ID', 0)]
        else:
            selected_gpus = sorted({
                globals().get('MODEL_GPU_ID', 0),
                globals().get('TEXT_GPU_ID', 1) if REMOTE_SERVICE_ID.startswith(('06_MiniMax_', '06_HyperFlow_'))
                else globals().get('AUX_GPU_ID', 1) if REMOTE_SERVICE_ID.startswith(('06_HeartMuLa_', '06_LTX-2.5_', '06_Z-Image-Turbo_'))
                else globals().get('TEXT_GPU_ID', 1) if qwen else 0
            }) if not qwen else sorted({globals().get('GPU_ID', 0), globals().get('TEXT_GPU_ID', 1)})
        available = torch.cuda.device_count()
        if any(not isinstance(index, int) or index < 0 or index >= available for index in selected_gpus):
            return False, f'需要 cuda:{selected_gpus}，但實際只有 {available} 張 CUDA GPU。', []
        devices = [f'cuda:{index} {torch.cuda.get_device_name(index)}' for index in selected_gpus]
        if qwen:
            state = globals().get('QWEN21_STATE', {})
            if state.get('pipeline') is None or state.get('manager') is None:
                return False, 'Qwen Pipeline／GPU 記憶體管理器尚未初始化，請先執行模型初始化 Cell。', devices
            if state.get('device') != f"cuda:{globals().get('GPU_ID', 0)}":
                return False, 'Qwen Transformer 不在設定的 GPU 上。', devices
        elif REMOTE_SERVICE_ID.startswith('06_HeartMuLa_'):
            pipeline = globals().get('pipe')
            if pipeline is None or pipeline.lazy_load or pipeline._mula is None or pipeline._codec is None:
                return False, 'HeartMuLa 雙 GPU Pipeline 尚未完整載入，請先執行模型載入 Cell。', devices
            for name, module, index in (('HeartMuLa', pipeline.mula, globals().get('MODEL_GPU_ID', 0)), ('HeartCodec', pipeline.codec, globals().get('AUX_GPU_ID', 1))):
                parameter = next(module.parameters(), None)
                if parameter is None or parameter.device != torch.device(f'cuda:{index}'):
                    return False, f'{name} 未位於指定的 cuda:{index}，請重新載入模型。', devices
        elif yue:
            gpu_id = globals().get('GPU_ID', 0)
            process = globals().get('SERVER_PROCS', {}).get(gpu_id)
            if process is None or process.poll() is not None:
                return False, f'YuE2 CUDA{gpu_id} 常駐服務尚未啟動。', devices
            if f'CUDA{gpu_id}' not in [str(value) for value in process.args]:
                return False, 'YuE2 常駐服務沒有使用預期的 CUDA 啟動參數。', devices
        else:
            base = globals().get('BASE')
            if not base and REMOTE_IS_LTX:
                base = f"http://127.0.0.1:{globals().get('COMFY_PORT', 8188)}"
            if not isinstance(base, str) or not base:
                return False, 'ComfyUI 尚未啟動，請先執行模型服務 Cell。', devices
            with urllib.request.urlopen(base.rstrip('/') + '/system_stats', timeout=4) as response:
                stats = json.load(response)
            comfy_devices = stats.get('devices', [])
            if not any(str(item.get('type', '')).lower().startswith('cuda') or
                       str(item.get('name', '')).lower().startswith('cuda:')
                       for item in comfy_devices if isinstance(item, dict)):
                return False, 'ComfyUI 沒有回報 CUDA 裝置，已拒絕 CPU-only 推論。', devices
        return True, 'CUDA 後端已就緒（不代表所有權重常駐 GPU 記憶體）。', devices
    except Exception as error:
        return False, f'GPU 檢查失敗：{type(error).__name__}: {error}', []

'''
Verify every global required by the selected Notebook generation engine.
驗證所選模型的生成程式所需變數與設定已完成初始化。
'''
def remote_inference_ready(task):
    namespace = globals()
    common = {
        '06_HeartMuLa_': ('ROOT', 'pipe'),
        '06_YuE2_': ('ROOT', 'GPU_ID', 'PORTS', 'SERVER_PROCS'),
        '06_Z-Image-Turbo_': ('ROOT', 'BASE', 'LOG_PATH'),
        '06_Qwen_Image_': ('MODEL_MODE', 'GPU0_MODEL_GIB', 'GPU_ID', 'QWEN21_STATE'),
        '06_LTX-2.5_': ('ROOT', 'ENGINE_PATH', 'COMFY_PORT', 'COMFY_LOG', 'COMFY_PROC',
                          'ltx_available_ram_gib', 'LTX_MIN_AVAILABLE_RAM_GIB', 'importlib'),
        '06_MiniMax_H3_': ('ROOT', 'BASE', 'COMFY', 'LOG', 'PORT', 'TASK_CONFIGS',
                              'H3_DESIRED_CONFIG', 'H3_ACTIVE_CONFIG', 'H3_MODEL_CHOICE',
                              'H3_ACTIVE_MODEL', 'DIT_NAME', 'TEXT_MODE', 'TEXT_GPU',
                              'QUANT_SIZES_GIB', 'ready', 'choices', 'check_choice'),
        '06_HyperFlow_H3_': ('ROOT', 'BASE', 'COMFY', 'LOG', 'PORT', 'TASK_CONFIGS',
                                'H3_DESIRED_CONFIG', 'H3_ACTIVE_CONFIG', 'H3_MODEL_CHOICE',
                                'H3_ACTIVE_MODEL', 'DIT_NAME', 'TEXT_MODE', 'TEXT_GPU',
                                'QUANT_SIZES_GB', 'ready', 'choices', 'ENCODERS',
                                'ENCODER_FILE', 'MODEL_FAMILY', 'WEIGHTS_IN',
                                'TEXT_LOADER_NODE', 'PROJECTION_FILE'),
    }
    required = next((names for prefix, names in common.items()
                     if REMOTE_SERVICE_ID.startswith(prefix)), ())
    missing = [name for name in required if name not in namespace or namespace[name] is None]
    if missing:
        return False, 'Notebook 尚未完成初始化，請先執行對應的模型／服務 Cells；缺少：' + ', '.join(missing)
    if task not in REMOTE_ALLOWED_TASKS:
        return False, '此模型不支援生成模式：' + task
    if REMOTE_SERVICE_ID.startswith(('06_MiniMax_H3_', '06_HyperFlow_H3_')):
        config = namespace['TASK_CONFIGS'].get(task)
        if not isinstance(config, dict):
            return False, f'{task.upper()} 尚未設定；請先執行 Notebook 對應的模式設定 Cell。'
        if namespace['H3_ACTIVE_CONFIG'] != namespace['H3_DESIRED_CONFIG']:
            return False, 'GPU／Offload 設定與已啟動 ComfyUI 不一致，請重新啟動模型服務 Cell。'
        quant = config.get('quant')
        if REMOTE_SERVICE_ID.startswith('06_MiniMax_H3_'):
            variant = 'ref2va_pruned' if task == 'r2v' else 'fl2va_pruned'
            expected = (variant, quant)
            expected_file = f'minimax_h3_{variant}-{quant}.gguf'
        else:
            variant = 'ref2va' if task == 'r2v' else 'fl2va'
            encoder = config.get('encoder_variant')
            expected = (variant, quant, encoder)
            expected_file = f'minimax_h3_{variant}_hyperflow_flash_14b-{quant}.gguf'
            if encoder not in namespace['ENCODERS']:
                return False, f'{task.upper()} 的 Encoder 設定無效，請重新執行模式設定 Cell。'
            if Path(namespace['ENCODER_FILE']).name != namespace['ENCODERS'][encoder][1]:
                return False, 'GPU 1 Encoder 與此模式不一致，請重新執行模型下載／啟動 Cells。'
        if namespace['H3_MODEL_CHOICE'] != expected or namespace['H3_ACTIVE_MODEL'] != expected:
            return False, f'{task.upper()} 尚未載入相應模型權重；請設定模式並重新執行模型下載／啟動 Cells。'
        if namespace['DIT_NAME'] != expected_file:
            return False, 'ComfyUI 使用的模型檔名與任務不一致，請重新載入正確模型。'
    elif REMOTE_SERVICE_ID.startswith('06_Qwen_Image_'):
        state = namespace['QWEN21_STATE']
        if state.get('mode') != namespace['MODEL_MODE']:
            return False, 'Qwen MODEL_MODE 與已初始化的模型模式不同，請重新執行模型初始化 Cell。'
        if state.get('gpu0_model_gib') != namespace['GPU0_MODEL_GIB']:
            return False, 'Qwen GPU0_MODEL_GIB 與目前載入設定不同，請重新初始化 GPU 記憶體管理器。'
    elif REMOTE_IS_LTX:
        if not Path(namespace['ENGINE_PATH']).is_file():
            return False, 'LTX 模型引擎檔案不存在；請先執行模型與 Workflow Cell。'
        if namespace['COMFY_PROC'].poll() is not None:
            return False, 'LTX ComfyUI 行程已停止，請重新執行啟動 ComfyUI Cell。'
    return True, '模型初始化與所選模式已就緒。'

'''
Validate client parameters before they occupy a FIFO slot.
在進入 FIFO 前檢查 client 參數，避免排隊後才因設定錯誤而失敗。
'''
def remote_validate_fields(task, fields):
    required = {'task'}
    if REMOTE_SERVICE_ID.startswith('06_HeartMuLa_'):
        required.update(('lyrics', 'tags', 'duration', 'seed'))
    elif REMOTE_SERVICE_ID.startswith('06_YuE2_'):
        required.update(('style', 'lyrics', 'duration', 'steps', 'lm_seed', 'seed', 'cot'))
    elif REMOTE_KIND == 'image_text':
        required.update(('prompt', 'width', 'height', 'steps', 'seed', 'cfg'))
    elif REMOTE_KIND == 'image_multi':
        required.update(('prompt', 'negative_prompt', 'sampler', 'reference_size',
                         'width', 'height', 'steps', 'seed', 'cfg'))
    elif REMOTE_IS_LTX:
        required.update(('prompt', 'width', 'height', 'seconds', 'seed', 'image_strength'))
    else:
        required.update(('prompt', 'width', 'height', 'seconds', 'steps', 'seed', 'ref_image_size'))
        if REMOTE_SERVICE_ID.startswith('06_MiniMax_H3_'):
            required.add('audio_shift')
    missing = required - fields.keys()
    if missing:
        raise ValueError('請求缺少必要參數：' + ', '.join(sorted(missing)))
    integer_fields = ('seed', 'lm_seed', 'width', 'height', 'steps', 'duration', 'reference_size')
    float_fields = ('seconds', 'cfg', 'image_strength', 'audio_shift')
    numbers = {}
    for key in integer_fields:
        if key in fields:
            raw = fields[key]
            if isinstance(raw, bool) or str(raw).strip() != str(int(raw)):
                raise ValueError(f'{key} 必須是整數')
            numbers[key] = int(raw)
    for key in float_fields:
        if key in fields:
            numbers[key] = float(fields[key])
            if not math.isfinite(numbers[key]):
                raise ValueError(f'{key} 必須是有限數值')
    for key in ('seed', 'lm_seed'):
        if key in numbers and not -1 <= numbers[key] < 2**32:
            raise ValueError(f'{key} 必須介於 -1 至 4294967295')
    for key in ('width', 'height'):
        if key in numbers:
            size = numbers[key]
            divisor = 64 if REMOTE_KIND == 'image_text' else 32
            if size < 1 or size > 4096 or size % divisor:
                raise ValueError(f'{key} 必須介於 1～4096 且為 {divisor} 的倍數')
    for key in ('seconds', 'duration'):
        if key in numbers and not 0 < numbers[key] <= 3600:
            raise ValueError(f'{key} 必須大於 0 且不超過 3600')
    if 'steps' in numbers and not 1 <= numbers['steps'] <= 200:
        raise ValueError('steps 必須介於 1～200')
    if REMOTE_SERVICE_ID.startswith('06_HyperFlow_H3_') and numbers.get('steps') not in (4, 6, 8):
        raise ValueError('HyperFlow steps 只支援 4、6、8')
    if 'image_strength' in numbers and not 0 <= numbers['image_strength'] <= 1:
        raise ValueError('image_strength 必須介於 0～1')
    if 'reference_size' in numbers and not 64 <= numbers['reference_size'] <= 2048:
        raise ValueError('reference_size 必須介於 64～2048')
    if 'audio_shift' in numbers and not 0.01 <= numbers['audio_shift'] <= 100:
        raise ValueError('audio_shift 必須介於 0.01～100')
    if fields.get('ref_image_size', 'match') not in ('match', 'max'):
        raise ValueError('ref_image_size 只支援 match 或 max')
    if REMOTE_KIND in ('video', 'image_text', 'image_multi') and not str(fields.get('prompt', '')).strip():
        raise ValueError('PROMPT 不可為空白')
    if task == 'r2v':
        matches = [int(number) for number in re.findall(r'<Picture\s+(\d+)>', str(fields['prompt']), re.I)]
        if any(number < 1 or number > int(fields.get('_image_count', 0)) for number in matches):
            raise ValueError('R2V Prompt 的 <Picture N> 編號超出已上傳的圖片數量')

'''
Report whether the model is ready for CUDA inference.
回報模型是否已具備 CUDA 推論條件。
'''
def remote_backend_ready():
    return remote_gpu_status()[0]

'''
A Python thread cannot safely terminate an already running CUDA inference.
Python thread 無法安全強制終止正在執行的 CUDA 推論。
'''
def remote_stalled():
    with REMOTE_JOB_LOCK:
        return any(record.get('status') == 'running' and
                   time.monotonic() - record.get('_start_clock', time.monotonic()) > REMOTE_STALL_SECONDS
                   for record in REMOTE_JOBS.values())

'''
Remove obsolete input files once the model no longer needs them.
模型不再需要輸入檔案時立即移除，避免參考圖片長期佔用空間。
'''
def remote_remove_inputs(job_id):
    folder = REMOTE_ROOT / job_id
    for file in folder.glob('*'):
        if file.name.startswith('input.') or (file.stem.isdecimal() and file.suffix.lower() in {'.png', '.jpg', '.jpeg', '.webp'}):
            file.unlink(missing_ok=True)

'''
Keep only finished results; never delete active jobs or in-progress downloads.
僅清理已結束工作，絕不刪除排隊、推論或正在下載的檔案。
'''
def remote_cleanup():
    now = time.time()
    with REMOTE_JOB_LOCK:
        finished = [(job_id, record) for job_id, record in REMOTE_JOBS.items()
                    if record.get('status') in ('done', 'error', 'expired') and
                    not record.get('_downloads')]
        finished.sort(key=lambda item: item[1].get('finished_at', now))
        sizes = {}
        for job_id, record in finished:
            folder = REMOTE_ROOT / job_id
            sizes[job_id] = sum(path.stat().st_size for path in folder.glob('**/*') if path.is_file()) if folder.exists() else 0
        total = sum(sizes.values())
        for job_id, record in finished:
            old = now - record.get('finished_at', now) > REMOTE_RESULT_TTL
            if old or total > REMOTE_RESULT_BYTES:
                shutil.rmtree(REMOTE_ROOT / job_id, ignore_errors=True)
                total -= sizes[job_id]
                record.update(status='expired', error='結果已過期或因儲存空間不足而清除，請重新提交。')
                record.pop('_path', None)
                record.pop('_last_frame', None)
            if record['status'] == 'expired' and now - record.get('finished_at', now) > 2 * REMOTE_RESULT_TTL:
                REMOTE_JOBS.pop(job_id, None)

'''
Sweep completed records periodically without interfering with inference.
定期整理已完成資料，不干擾單一 GPU worker 的推論。
'''
def remote_maintenance():
    while True:
        try:
            remote_cleanup()
        except Exception as error:
            print('API cleanup warning:', type(error).__name__, str(error)[:200])
        time.sleep(300)

'''
Serve only the generation API without exposing notebook controls.
只提供生成 API，不公開 Notebook 的管理功能。
'''
class RemoteHandler(BaseHTTPRequestHandler):
    def log_message(self, fmt, *args):
        return

    def send_json(self, code, payload, retry_after=None):
        data = json.dumps(payload, ensure_ascii=False).encode('utf-8')
        self.send_response(code)
        self.send_header('Content-Type', 'application/json; charset=utf-8')
        self.send_header('Content-Length', str(len(data)))
        if retry_after is not None:
            self.send_header('Retry-After', str(retry_after))
        self.end_headers()
        self.wfile.write(data)

    def do_GET(self):
        path = urlsplit(self.path).path
        if path == '/health':
            with REMOTE_JOB_LOCK:
                running = sum(record.get('status') == 'running' for record in REMOTE_JOBS.values())
            gpu_ready, gpu_message, gpu_devices = remote_gpu_status()
            requested_task = parse_qs(urlsplit(self.path).query).get('task', [REMOTE_ALLOWED_TASKS[0]])[-1]
            inference_ready, inference_message = remote_inference_ready(requested_task)
            if gpu_ready and not inference_ready:
                gpu_message = inference_message
            self.send_json(200, {'ok': True, 'api': 'kaggle-generation-bridge-v3',
                                 'service_id': REMOTE_SERVICE_ID, 'task': requested_task,
                                 'backend_ready': gpu_ready and inference_ready, 'gpu_ready': gpu_ready,
                                 'gpu_message': gpu_message, 'gpu_devices': gpu_devices,
                                 'stalled': remote_stalled(), 'running': running,
                                 'waiting': REMOTE_QUEUE.qsize(), 'queue_capacity': REMOTE_QUEUE.maxsize})
            return
        if path == '/schema':
            self.send_json(200, {'kind': REMOTE_KIND, 'tasks': REMOTE_ALLOWED_TASKS,
                                 'fields': REMOTE_FIELDS, 'image_limit': REMOTE_MAX_IMAGES,
                                 'upload_field': 'images' if REMOTE_MAX_IMAGES else None,
                                 'queue_capacity': REMOTE_QUEUE.maxsize,
                                 'model_switching': 'configure in notebook first'})
            return
        pieces = path.strip('/').split('/')
        if pieces[0] != 'jobs' or len(pieces) not in (2, 3):
            self.send_json(404, {'error': 'Not found'})
            return
        job_id = pieces[1]
        with REMOTE_JOB_LOCK:
            record = REMOTE_JOBS.get(job_id)
            item = dict(record) if record else None
            if item and item['status'] == 'running':
                item['stalled'] = time.monotonic() - item.get('_start_clock', time.monotonic()) > REMOTE_STALL_SECONDS
            if item and item['status'] == 'queued':
                waiting = [key for key, value in REMOTE_JOBS.items() if value.get('status') == 'queued']
                if job_id in waiting:
                    item['queue_position'] = waiting.index(job_id) + 1
            if item and len(pieces) == 3 and pieces[2] in ('file', 'last-frame') and item['status'] == 'done':
                record['_downloads'] = record.get('_downloads', 0) + 1
                downloading = True
            else:
                downloading = False
        if item is None:
            self.send_json(404, {'error': 'Unknown job'})
            return
        if len(pieces) == 2:
            for field in tuple(item):
                if field.startswith('_'):
                    item.pop(field)
            self.send_json(200, item)
            return
        if not downloading:
            self.send_json(404, {'error': 'Result not available for this job'})
            return
        try:
            file_path = item.get('_path') if pieces[2] == 'file' else item.get('_last_frame')
            if not file_path or not Path(file_path).is_file():
                self.send_json(404, {'error': 'Output is no longer available'})
                return
            result = Path(file_path)
            self.send_response(200)
            self.send_header('Content-Type', 'application/octet-stream')
            self.send_header('Content-Disposition', f'attachment; filename="{result.name}"')
            self.send_header('Content-Length', str(result.stat().st_size))
            self.end_headers()
            with result.open('rb') as stream:
                shutil.copyfileobj(stream, self.wfile, length=1024 * 1024)
        except (BrokenPipeError, ConnectionResetError):
            pass
        finally:
            with REMOTE_JOB_LOCK:
                record['_downloads'] -= 1

    def do_POST(self):
        if urlsplit(self.path).path != '/generate':
            self.send_json(404, {'error': 'Not found'})
            return
        supplied_id = self.headers.get('X-Request-ID', '').strip()
        try:
            job_id = uuid.UUID(supplied_id).hex if supplied_id else uuid.uuid4().hex
        except ValueError:
            self.send_json(400, {'error': 'X-Request-ID must be a UUID'})
            return
        with REMOTE_JOB_LOCK:
            existing = REMOTE_JOBS.get(job_id)
        if existing:
            self.send_json(202, {'job_id': job_id, 'status_url': f'/jobs/{job_id}',
                                 'download_url': f'/jobs/{job_id}/file', 'reused': True})
            return
        if remote_stalled():
            self.send_json(503, {'error': 'GPU worker appears stalled; inspect the Notebook before accepting more jobs'}, 60)
            return
        gpu_ready, gpu_message, _ = remote_gpu_status()
        if not gpu_ready:
            self.send_json(409, {'error': gpu_message})
            return
        if shutil.disk_usage(REMOTE_ROOT).free < REMOTE_MIN_FREE_BYTES:
            self.send_json(507, {'error': 'Kaggle disk space is low; clean old outputs before retrying'})
            return
        if not REMOTE_UPLOAD_SLOTS.acquire(timeout=1):
            self.send_json(503, {'error': 'Upload slots busy; retry later'}, 5)
            return
        staging = None
        try:
            self.connection.settimeout(120)
            length = int(self.headers.get('Content-Length', '0'))
            if not 0 < length <= 120 * 1024 * 1024:
                raise ValueError('Request must be between 1 byte and 120 MiB')
            body = self.rfile.read(length)
            if len(body) != length:
                raise ValueError('Incomplete request body')
            content_type = self.headers.get('Content-Type', '')
            fields, files = {}, []
            if content_type.startswith('application/json'):
                fields = json.loads(body)
            elif content_type.startswith('application/x-www-form-urlencoded'):
                fields = {key: values[-1] for key, values in parse_qs(body.decode('utf-8')).items()}
            elif content_type.startswith('multipart/form-data'):
                message = BytesParser(policy=policy.default).parsebytes(
                    b'Content-Type: ' + content_type.encode() + b'\r\nMIME-Version: 1.0\r\n\r\n' + body)
                for part in message.iter_parts():
                    name = part.get_param('name', header='content-disposition')
                    data = part.get_payload(decode=True)
                    if name == 'images':
                        files.append((part.get_filename() or 'upload.png', data))
                    elif name:
                        fields[name] = data.decode('utf-8')
            else:
                raise ValueError('Use application/json, form data, or multipart/form-data')
            if not isinstance(fields, dict):
                raise ValueError('Parameters must be a JSON object')
            if 'abc' in fields and len(str(fields['abc']).encode('utf-8')) > 1024 * 1024:
                raise ValueError('ABC score must be 1 MiB or smaller')
            task = str(fields.get('task', REMOTE_ALLOWED_TASKS[0])).lower()
            if task not in REMOTE_ALLOWED_TASKS:
                raise ValueError('Unsupported task: ' + task)
            if len(files) > REMOTE_MAX_IMAGES:
                raise ValueError(f'Only {REMOTE_MAX_IMAGES} input images are supported')
            if REMOTE_KIND in ('music', 'image_text') and files:
                raise ValueError('This model does not accept reference images')
            if REMOTE_KIND == 'video':
                minimum, maximum = {'t2v': (0, 0), 'i2v': (1, 1), 'flf2v': (2, 2),
                                    'r2v': (1, REMOTE_MAX_IMAGES)}[task] if task == 'r2v' else {
                                    't2v': (0, 0), 'i2v': (1, 1), 'flf2v': (2, 2)}[task]
                if not minimum <= len(files) <= maximum:
                    raise ValueError('Input image count does not match the selected video task')
            if REMOTE_KIND == 'image_multi':
                minimum, maximum = {'text': (0, 0), 'edit': (1, 1), 'multi': (2, REMOTE_MAX_IMAGES)}[task]
                if not minimum <= len(files) <= maximum:
                    raise ValueError(f'Task {task} requires {minimum}–{maximum} images')
            for key in fields:
                if key not in REMOTE_FIELDS:
                    raise ValueError('Unsupported parameter: ' + key)
            remote_validate_fields(task, dict(fields, _image_count=len(files)))
            inference_ready, inference_message = remote_inference_ready(task)
            if not inference_ready:
                self.send_json(409, {'error': inference_message})
                return
            for filename, data in files:
                if not isinstance(data, bytes) or len(data) > 20 * 1024 * 1024:
                    raise ValueError('Each image must be 20 MiB or smaller')
                if Path(filename).suffix.lower() not in {'.png', '.jpg', '.jpeg', '.webp'}:
                    raise ValueError('Only PNG/JPEG/WebP images are supported')
                from PIL import Image
                with Image.open(io.BytesIO(data)) as photo:
                    photo.verify()
            staging = Path(tempfile.mkdtemp(prefix='pending_', dir=REMOTE_ROOT))
            image_names = []
            for index, (filename, data) in enumerate(files, 1):
                target = staging / f'{index:03d}{Path(filename).suffix.lower()}'
                target.write_bytes(data)
                image_names.append(target.name)
            with REMOTE_JOB_LOCK:
                existing = REMOTE_JOBS.get(job_id)
                full = REMOTE_QUEUE.full()
                if not existing and not full:
                    folder = REMOTE_ROOT / job_id
                    staging.rename(folder)
                    staging = None
                    image_paths = [str(folder / name) for name in image_names]
                    REMOTE_JOBS[job_id] = {'id': job_id, 'task': task, 'status': 'queued',
                                           'created_at': time.time()}
                    REMOTE_QUEUE.put_nowait((job_id, fields, image_paths))
                    position = REMOTE_QUEUE.qsize()
            if existing:
                self.send_json(202, {'job_id': job_id, 'status_url': f'/jobs/{job_id}',
                                     'download_url': f'/jobs/{job_id}/file', 'reused': True})
            elif full:
                self.send_json(429, {'error': 'Queue is full; retry later'}, 15)
            else:
                self.send_json(202, {'job_id': job_id, 'status_url': f'/jobs/{job_id}',
                                     'download_url': f'/jobs/{job_id}/file', 'queue_position': position})
        except (ValueError, KeyError, TypeError, json.JSONDecodeError) as error:
            self.send_json(400, {'error': str(error)})
        except (TimeoutError, ConnectionError, OSError) as error:
            self.send_json(503, {'error': f'Upload temporarily failed: {type(error).__name__}'}, 10)
        except Exception as error:
            self.send_json(500, {'error': type(error).__name__ + ': ' + str(error)[:250]})
        finally:
            if staging is not None:
                shutil.rmtree(staging, ignore_errors=True)
            REMOTE_UPLOAD_SLOTS.release()

'''
Convert numeric HTTP form fields into explicitly supported generator settings.
只轉換白名單中的 HTTP 參數，不執行客戶端提供的任何程式碼。
'''
def remote_generate(job_id, fields, image_paths):
    with REMOTE_ENGINE_LOCK:
        with REMOTE_JOB_LOCK:
            REMOTE_JOBS[job_id].update(status='running', _start_clock=time.monotonic())
        namespace = globals()
        keys = ['TASK', 'PROMPT', 'WIDTH', 'HEIGHT', 'SEED', 'LM_SEED', 'SECONDS', 'STEPS', 'CFG',
                'LYRICS', 'TAGS', 'STYLE', 'DURATION', 'GPU_ID', 'COT', 'ABC_PATH',
                'OUTPUT_NAME', 'REFERENCE_PATHS', 'ACTIVE_JOB', 'TASK_CONFIGS',
                'R2V_REF_DIR', 'NEGATIVE_PROMPT', 'SAMPLER', 'REFERENCE_SIZE',
                'OUTPUTS', 'output', 'OUTPUT_PATH', 'OUTPUT_MP4', 'LAST_OUTPUT', 'RUN_SEED']
        before = {key: copy.deepcopy(namespace.get(key)) for key in keys if key in namespace}
        missing = set(keys) - set(before)
        try:
            task = fields.get('task', REMOTE_ALLOWED_TASKS[0]).lower()
            numerics = {'seed': int, 'lm_seed': int, 'width': int, 'height': int, 'steps': int,
                        'seconds': float, 'cfg': float, 'duration': int,
                        'image_strength': float, 'reference_size': int, 'audio_shift': float}
            inputs = {}
            for key, value in fields.items():
                inputs[key] = numerics[key](value) if key in numerics else value

            '''
            Resolve -1 once on the server so the exact value can be reproduced.
            在服務端一次決定 -1 的實際種子，方便重現結果。
            '''
            for seed_key in ('seed', 'lm_seed'):
                if seed_key in inputs:
                    if not -1 <= inputs[seed_key] < 2**32:
                        raise ValueError(f'{seed_key} must be -1 or between 0 and 4294967295')
                    if inputs[seed_key] == -1:
                        inputs[seed_key] = secrets.randbelow(2**32)
            with REMOTE_JOB_LOCK:
                for seed_key in ('seed', 'lm_seed'):
                    if seed_key in inputs:
                        REMOTE_JOBS[job_id][seed_key + '_used'] = inputs[seed_key]
            for dimension in ('width', 'height'):
                if dimension in inputs and not 0 < inputs[dimension] <= 4096:
                    raise ValueError('Width and height must be between 1 and 4096')
            for key in ('seconds', 'duration'):
                if key in inputs and not 0 < inputs[key] <= 3600:
                    raise ValueError('Duration must be between 0 and 3600 seconds')
            if 'reference_size' in inputs and not 64 <= inputs['reference_size'] <= 2048:
                raise ValueError('Reference size must be between 64 and 2048')
            if 'ref_image_size' in inputs and inputs['ref_image_size'] not in ('match', 'max'):
                raise ValueError('Reference image mode must be match or max')
            if 'image_strength' in inputs and not 0 <= inputs['image_strength'] <= 1:
                raise ValueError('Image strength must be between 0 and 1')
            if 'steps' in inputs and not 1 <= inputs['steps'] <= 200:
                raise ValueError('Steps must be between 1 and 200')
            if REMOTE_KIND == 'video':
                if 'TASK_CONFIGS' in namespace:
                    if task not in namespace['TASK_CONFIGS']:
                        raise ValueError('Run this task configuration cell before API generation')
                    updated = copy.deepcopy(namespace['TASK_CONFIGS'][task])
                    for field in ('prompt', 'width', 'height', 'seconds', 'steps', 'seed',
                                  'audio_shift', 'ref_image_size'):
                        if field in inputs:
                            updated[field] = inputs[field]
                    updated['output_name'] = f'remote_{job_id[:12]}.mp4'
                    if task == 'i2v':
                        updated['first_image'] = image_paths[0]
                    elif task == 'flf2v':
                        updated['first_image'], updated['last_image'] = image_paths
                    elif task == 'r2v':
                        updated['ref_images'] = image_paths
                        namespace['R2V_REF_DIR'] = Path(image_paths[0]).parent
                    namespace['TASK_CONFIGS'][task] = updated
                    namespace['TASK'] = task
                else:
                    namespace['ACTIVE_JOB'] = {'mode': task, 'prompt': inputs.get('prompt', namespace.get('PROMPT', '')),
                                               'run_name': f'remote_{job_id[:12]}',
                                               'first_path': image_paths[0] if image_paths else None,
                                               'last_path': image_paths[1] if len(image_paths) > 1 else None}
                    for key in ('width', 'height', 'seconds', 'seed', 'image_strength'):
                        if key in inputs:
                            namespace[key.upper()] = inputs[key]
            else:
                for key in ('prompt', 'width', 'height', 'seed', 'lm_seed', 'steps', 'cfg',
                            'lyrics', 'tags', 'style', 'duration', 'negative_prompt',
                            'sampler', 'reference_size', 'cot'):
                    if key in inputs:
                        namespace[key.upper()] = inputs[key]
                if REMOTE_KIND == 'image_multi':
                    namespace['TASK'] = task
                    namespace['REFERENCE_PATHS'] = image_paths
                if REMOTE_KIND == 'music':
                    namespace['OUTPUT_NAME'] = f'remote_{job_id[:12]}.mp3'
                    if 'ABC_PATH' in namespace:
                        namespace['ABC_PATH'] = ''
                    if 'abc' in inputs:
                        score = (REMOTE_ROOT / job_id / 'input.abc')
                        score.write_text(str(inputs['abc']), encoding='utf-8')
                        namespace['ABC_PATH'] = str(score)
                if REMOTE_KIND in ('image_multi', 'image_text'):
                    namespace['OUTPUT_NAME'] = f'remote_{job_id[:12]}.png'
            for output_key in ('OUTPUTS', 'output', 'OUTPUT_PATH', 'OUTPUT_MP4', 'LAST_OUTPUT'):
                namespace.pop(output_key, None)
            gpu_ready, gpu_message, _ = remote_gpu_status()
            if not gpu_ready:
                raise RuntimeError(gpu_message)
            inference_ready, inference_message = remote_inference_ready(task)
            if not inference_ready:
                raise RuntimeError(inference_message)
            exec(compile(REMOTE_SCRIPT, '<notebook-generation-cell>', 'exec'), namespace)
            paths = []
            if REMOTE_KIND == 'video':
                paths = [namespace.get('LAST_OUTPUT') or namespace.get('OUTPUT_MP4')]
            elif REMOTE_KIND == 'image_text':
                paths = [namespace.get('OUTPUT_PATH')]
            elif REMOTE_KIND == 'image_multi':
                paths = [namespace.get('output')]
            elif REMOTE_KIND == 'music':
                paths = namespace.get('OUTPUTS', []) if 'OUTPUTS' in namespace else [namespace.get('output')]
            results = [Path(path) for path in paths if path is not None]
            if REMOTE_KIND == 'music' and REMOTE_SERVICE_ID.startswith('06_YuE2_'):
                results += [sidecar for path in list(results)
                            for ext in ('.json', '.abc')
                            if (sidecar := path.with_suffix(ext)).is_file()]
            if not results or any(not path.is_file() for path in results):
                raise FileNotFoundError('Generation finished without a saved result file')
            folder = REMOTE_ROOT / job_id
            if len(results) == 1:
                target = folder / f'result{results[0].suffix.lower()}'
                shutil.copy2(results[0], target)
            else:
                target = folder / 'results.zip'
                with zipfile.ZipFile(target, 'w', compression=zipfile.ZIP_STORED) as archive:
                    for result in results:
                        archive.write(result, arcname=result.name)
            last_frame = None
            if REMOTE_KIND == 'video' and shutil.which('ffprobe') and shutil.which('ffmpeg'):
                try:
                    probe = subprocess.run(['ffprobe', '-v', 'error', '-count_frames',
                                            '-select_streams', 'v:0', '-show_entries',
                                            'stream=nb_read_frames', '-of', 'json', str(target)],
                                           check=True, text=True, capture_output=True, timeout=120)
                    index = int(json.loads(probe.stdout)['streams'][0]['nb_read_frames']) - 1
                    if index >= 0:
                        last_frame = folder / 'last_frame.png'
                        subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', str(target),
                                        '-vf', rf'select=eq(n\,{index})', '-frames:v', '1',
                                        str(last_frame)], check=True)
                except (OSError, ValueError, KeyError, subprocess.CalledProcessError, subprocess.TimeoutExpired):
                    last_frame = None

            '''
            Remove only source files bearing this remote job's unique prefix.
            僅清理本次遠端工作專屬原始輸出，保留 Notebook 手動生成的檔案。
            '''
            for original in results:
                if original.name.startswith(f'remote_{job_id[:12]}') and original != target:
                    try:
                        original.unlink(missing_ok=True)
                    except OSError:
                        pass
            comfy = globals().get('COMFY')
            if comfy is not None and (Path(comfy) / 'output').exists():
                for artifact in (Path(comfy) / 'output').rglob(f'remote_{job_id[:12]}*'):
                    if artifact.is_file():
                        try:
                            artifact.unlink()
                        except OSError:
                            pass
            with REMOTE_JOB_LOCK:
                REMOTE_JOBS[job_id].update(status='done', filename=target.name,
                                           bytes=target.stat().st_size, _path=str(target),
                                           last_frame_available=bool(last_frame),
                                           _last_frame=str(last_frame) if last_frame else None)
        except Exception as error:
            with REMOTE_JOB_LOCK:
                REMOTE_JOBS[job_id].update(status='error', error=f'{type(error).__name__}: {error}'[:2000])
        finally:
            for key in missing:
                namespace.pop(key, None)
            namespace.update(before)

'''
Use one GPU worker, regardless of the number of HTTP request threads.
不論有多少 HTTP 連線，都只有一個 GPU worker 消費佇列。
'''
def remote_worker():
    while True:
        job_id, fields, image_paths = REMOTE_QUEUE.get()
        try:
            remote_generate(job_id, fields, image_paths)
        except Exception as error:
            with REMOTE_JOB_LOCK:
                if job_id in REMOTE_JOBS:
                    REMOTE_JOBS[job_id].update(status='error', error=f'{type(error).__name__}: {error}'[:2000])
        finally:
            with REMOTE_JOB_LOCK:
                if job_id in REMOTE_JOBS:
                    REMOTE_JOBS[job_id]['finished_at'] = time.time()
            try:
                remote_remove_inputs(job_id)
            except OSError as error:
                print('API input cleanup warning:', str(error)[:180])
            finally:
                REMOTE_QUEUE.task_done()

if not (globals().get('REMOTE_WORKER') and REMOTE_WORKER.is_alive()):
    REMOTE_WORKER = threading.Thread(target=remote_worker, daemon=True)
    REMOTE_WORKER.start()

if not (globals().get('REMOTE_MAINTENANCE') and REMOTE_MAINTENANCE.is_alive()):
    REMOTE_MAINTENANCE = threading.Thread(target=remote_maintenance, daemon=True)
    REMOTE_MAINTENANCE.start()

old_tunnel = globals().get('REMOTE_NGROK_PROC')
if old_tunnel is not None and old_tunnel.poll() is None:
    old_tunnel.terminate()
    try:
        old_tunnel.wait(timeout=5)
    except subprocess.TimeoutExpired:
        old_tunnel.kill()
        old_tunnel.wait(timeout=5)

NGROK_URL = ''

'''
Install the official ngrok agent only if no usable binary exists.
系統尚無 ngrok 才下載官方執行檔，不重複安裝。
'''
ngrok_bin = shutil.which('ngrok') or str(ROOT / 'ngrok')
if not Path(ngrok_bin).is_file():
    url = 'https://bin.ngrok.com/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.tgz'
    with urllib.request.urlopen(url, timeout=90) as response:
        blob = response.read()
    with tarfile.open(fileobj=io.BytesIO(blob), mode='r:gz') as archive:
        entry = next((member for member in archive.getmembers() if Path(member.name).name == 'ngrok' and member.isfile()), None)
        if entry is None:
            raise RuntimeError('ngrok 官方下載檔案缺少執行檔')
        Path(ngrok_bin).write_bytes(archive.extractfile(entry).read())
    Path(ngrok_bin).chmod(0o755)

ngrok_token = getpass('輸入 ngrok Authtoken：').strip()
if not ngrok_token:
    raise ValueError('未輸入 ngrok Authtoken')

result = subprocess.run([ngrok_bin, 'config', 'add-authtoken', ngrok_token], capture_output=True, text=True)
if result.returncode:
    raise RuntimeError('ngrok token 設定失敗（請確認 Token 是否有效）')

del ngrok_token
result = subprocess.run([ngrok_bin, 'config', 'check'], capture_output=True, text=True)
if result.returncode:
    raise RuntimeError('ngrok 設定格式檢查失敗：' + result.stderr[-500:])

server = globals().get('REMOTE_SERVER')
server_thread = globals().get('REMOTE_SERVER_THREAD')
server_is_current = (
    server is not None
    and server_thread is not None
    and server_thread.is_alive()
    and getattr(server, 'notebook_id', None) == REMOTE_SERVICE_ID
)
if server is not None and not server_is_current:
    if server_thread is not None and server_thread.is_alive():
        server.shutdown()
    server.server_close()

if not server_is_current:
    try:
        REMOTE_SERVER = ThreadingHTTPServer(('127.0.0.1', REMOTE_PORT), RemoteHandler)
    except OSError as error:
        raise RuntimeError(f'本機 API Port {REMOTE_PORT} 已被其他服務占用') from error
    REMOTE_SERVER.notebook_id = REMOTE_SERVICE_ID
    REMOTE_SERVER.daemon_threads = True
    REMOTE_SERVER_THREAD = threading.Thread(target=REMOTE_SERVER.serve_forever, daemon=True)
    REMOTE_SERVER_THREAD.start()
    print('已啟動新的本機 API Server。')
else:
    REMOTE_SERVER.RequestHandlerClass = RemoteHandler
    print('已沿用本機 API Server；現有生成工作與結果保持不變。')

with urllib.request.urlopen(f'http://127.0.0.1:{REMOTE_PORT}/health', timeout=4) as response:
    if json.load(response).get('ok') is not True:
        raise RuntimeError('本機 API Server 尚未就緒')

'''
Use the agent inspection API to obtain the actual public URL.
透過 ngrok 本機檢查 API 取得實際對外 HTTPS 網址。
'''
NGROK_LOG = REMOTE_ROOT / 'ngrok.log'
with NGROK_LOG.open('w') as log:
    REMOTE_NGROK_PROC = subprocess.Popen([ngrok_bin, 'http', str(REMOTE_PORT)], stdout=log,
                                         stderr=subprocess.STDOUT, start_new_session=True)

NGROK_URL = ''
for attempt in range(25):
    if REMOTE_NGROK_PROC.poll() is not None:
        break
    try:
        with urllib.request.urlopen('http://127.0.0.1:4040/api/tunnels', timeout=2) as response:
            tunnels = json.load(response).get('tunnels', [])
        NGROK_URL = next((item['public_url'] for item in tunnels
                          if item.get('public_url', '').startswith('https://') and
                          str(REMOTE_PORT) in item.get('config', {}).get('addr', '')), '')
        if NGROK_URL:
            break
    except (OSError, ValueError, KeyError):
        pass
    time.sleep(1)

if not NGROK_URL:
    raise RuntimeError('ngrok URL 建立失敗：\n' + NGROK_LOG.read_text(errors='replace')[-2500:])

print('ngrok HTTPS URL:', NGROK_URL)
print('提醒：公開的 ngrok API URL 不需要額外驗證，請只分享給授課學生。')
print('GET /health | GET /schema | POST /generate | GET /jobs/{id} | GET /jobs/{id}/file')
print('影片輸出另提供：GET /jobs/{id}/last-frame')
print('✓ 第 8 節完成：選用 FIFO／ngrok API 已啟動。')


9. （選用）遠端 API 與 client 使用說明


In [ ]:
'''
Query the active bridge for supported parameters and task names.
從正在執行的橋接服務讀取實際支援的參數與模式。
'''
import json
import urllib.request
request = urllib.request.Request(f'http://127.0.0.1:{REMOTE_PORT}/schema')
with urllib.request.urlopen(request, timeout=5) as response:
    schema = json.load(response)

print(json.dumps(schema, ensure_ascii=False, indent=2))
print('用法：POST NGROK_URL + "/generate"（JSON 或 multipart/images），取得 job_id；')
print('輪詢 GET /jobs/{job_id}；完成後下載 GET /jobs/{job_id}/file。')
print('本機 requests 範例：06_HeartMuLa_音樂生成_client.py（與本 Notebook 同名對應）。')
print('使用前只需在對應的 client.py 填入 API_URL。')
print('本服務使用單一 FIFO worker，遠端請求不會同時啟動多筆 GPU 推論。')
print('✓ 第 9 節完成：API Schema 和 client 說明已顯示。')
